# 🏗️ 쿠팡 물류창고 관제 시스템 핵심 코드 및 알고리즘 종합 가이드

본 Jupyter Notebook은 다중 로봇 물류창고 관제 시스템(Control Tower)의 4대 핵심 컴포넌트(**Docker 인프라, PostgreSQL DB 스키마, ROS 2 Control Tower 노드, FastAPI 대시보드**)의 상세 설계 사양, 핵심 알고리즘 및 시퀀스 흐름도(Mermaid Flowchart)를 종합 해설하는 **엔지니어링 가이드북**입니다.

---

## 🐳 1. Docker 기반 하이브리드 DB 인프라 (`docker-compose.yml`)

관제 시스템의 데이터 저장소는 성능 극대화 및 데이터 무결성 보장을 위해 **관계형 DB(PostgreSQL)**와 **인메모리 DB(Redis)**를 혼합하여 운용합니다.

### 📌 1.1 하이브리드 설계 아키텍처 및 핵심 이유

```mermaid
graph TD
    AMR[AMR 주행 제어기] -->|고주파 텔레메트리 10Hz| Redis[(Redis 캐시)]
    Sorter[적재/포장 로봇] -->|트랜잭션 데이터 1Hz| SQL[(PostgreSQL)]
    CT[Control Tower] <-->|ZSET 우선순위 스케줄링| Redis
    CT <-->|조인 및 이력 보존| SQL
    Dash[FastAPI Dashboard] <-- WebSocket (1.5s) --> Browser[웹 브라우저]
```

* **PostgreSQL (Port 5432)**:
  * **역할**: 마스터 데이터(로봇 목록, 바닥 격자 맵 정보) 관리 및 영구 트랜잭션 데이터(패키지 라이프사이클, 작업대 보관 이력) 보존.
  * **키 포인트**: 외래키(Foreign Key) 제약 조건을 활용해 작업대가 이동하더라도 상자의 적재 상태 정보가 불일치하지 않도록 보장.
* **Redis (Port 6379)**:
  * **역할**: 고주파수(10Hz+) AMR 상태 데이터 캐싱 및 Sorted Set(ZSET) 기반 우선순위 제어 명령 큐 관리.
  * **키 포인트**: 초당 수십 번 발생하는 AMR의 위치 좌표 갱신 연산이 디스크 I/O 병목을 유발하지 않도록 인메모리 처리하여 관제탑 성능을 30 FPS 이상 유지.

### 📌 1.2 데이터베이스 커넥션 풀 (Connection Pool) 알고리즘
* **문제점**: ROS 2는 멀티스레드 비동기 콜백 환경입니다. 각 스레드가 쿼리를 실행할 때마다 DB 연결을 맺고 끊으면 오버헤드가 발생하고 포트 고갈 현상이 생깁니다.
* **해결 알고리즘**: `psycopg2.pool.ThreadedConnectionPool`을 도입하여 일정 수(예: 최소 5개, 최대 20개)의 DB 커넥션을 미리 생성해 둡니다. 스레드가 쿼리를 요청하면 풀에서 유휴 커넥션을 빌려준 뒤, 처리가 끝나면 원자적으로 풀에 반환합니다.

In [ ]:
# Docker 컨테이너의 기동 상태 및 포트 매핑을 확인합니다.
!docker ps --format "table {{.Names}}\t{{.Ports}}\t{{.Status}}"

---

## 🗄️ 2. PostgreSQL 스키마 설계 및 이월 적재 초기화 (`init.sql` & `init_june_8th_state.py`)

데이터 정규화 및 영업일 전환(Day Transition)을 실현하는 DB 스키마 구조와 데이터 흐름 알고리즘입니다.

### 📌 2.1 데이터 정규화 (1:N 조인 설계) 키 포인트
* 기존에는 작업대 테이블(`workstations`)에 8개 슬롯 정보를 직접 컬럼(예: `slot_1_status` 등)으로 넣어 공간 낭비와 불일치 위험이 있었습니다.
* 개선 후에는 `packages` 테이블이 **`workstation_id` (외래키)**와 **`slot_number` (1~8)**를 소유하여, 해당 작업대에 종속된 패키지를 `JOIN` 형태로 쿼리합니다. 이로써 슬롯 개수가 변경(예: 6개, 12개)되어도 테이블 스키마를 변경할 필요가 없습니다.

### 📌 2.2 영업일 전환 및 이월 적재 (Day Transition & Carry-over) 알고리즘

```mermaid
sequenceDiagram
    participant CT as Control Tower (관제탑)
    participant DB as PostgreSQL
    participant RD as Redis

    Note over CT, RD: 오늘 영업일 완료 (오늘 날짜 미완료 패키지 = 0)
    CT->>DB: Daily Report 마크다운 보고서 자동 생성
    CT->>RD: system:day_status를 'PENDING_TRANSITION'으로 변경
    Note over CT: 관리자가 대시보드에서 다음 날 영업 시작 클릭
    CT->>RD: system:today_date를 하루 뒤 날짜로 업데이트
    CT->>DB: [이월 재배치 예약] 실행
    CT->>DB: 2번 라인 작업대 -> 1번 라인 Active 버퍼로 이송 예약
    CT->>DB: 3번 라인 작업대 -> 2번 라인 Active 버퍼로 이송 예약
    CT->>DB: 3번 라인에는 메인 창고 주차장 스팟에서 새 빈 작업대 공급
    Note over CT, DB: 이월 작업대는 기존 적재 슬롯(예: 5/8)부터 신규 상자를 누적 적재
```

### 📌 2.3 6월 8일 데모용 초기 상태 맵 데이터 검증
초기화 실행 시 `WS01`은 이미 8개 상자가 완충되어 출고 대기 구역(`stage_01`)에 존재하고, `WS02`(1번 라인 Active), `WS03`(2번 라인 Active)에는 각각 전날 쌓다 남은 **5개의 이월 상자**들이 적재된 채로 대기합니다.

In [ ]:
import psycopg2
import os

db_host = os.environ.get("POSTGRES_HOST", "localhost")
try:
    conn = psycopg2.connect(
        host=db_host,
        database="warehouse_db",
        user="rokey",
        password="rokey_pass",
        port=5432
    )
    cursor = conn.cursor()
    
    # 1. 각 라인별 작업대와 슬롯 적재 현황 조회 (조인 쿼리)
    query = """
    SELECT w.workstation_id, w.current_location, COUNT(p.package_id) as loaded_slots
    FROM workstations w
    LEFT JOIN packages p ON w.workstation_id = p.workstation_id AND p.status IN ('IN_WORKSTATION', 'IN_WAREHOUSE')
    GROUP BY w.workstation_id, w.current_location, w.status
    ORDER BY w.workstation_id;
    """
    cursor.execute(query)
    print("📊 작업대별 적재 상태 검증:")
    for row in cursor.fetchall():
        print(f"   * 작업대: {row[0]} | 위치: {row[1]:<15} | 적재된 슬롯 수: {row[2]}/8 개")
        
    cursor.close()
    conn.close()
except Exception as e:
    print(f"❌ DB 조회 실패: {e}")

---

## 🤖 3. ROS 2 Control Tower 노드 (`control_tower_node.py`)

관제탑 노드는 우선순위 큐(Redis ZSET)에 적재되는 명령을 스케줄링하고, 물리 충돌 방지 및 로봇 일시정지 상태 제어를 전담합니다.

### 📌 3.1 JIT (Just-In-Time) 일시정지 인터로킹 알고리즘
* **알고리즘 상세**:
  1. 적재 로봇(`sg2_in_XX`)이 4번째 상자 적재 완료 시 관제탑에 `ReportInboundProgress` 호출.
  2. 관제탑은 즉시 `/{robot_id}/pause_status` 토픽에 `data = True`를 발행하여 로봇을 일시정지 상태로 락(Lock)을 겁니다.
  3. 관제탑은 AMR에게 해당 작업대의 180도 회전 명령(`ROTATE_WORKSTATION`)을 하달합니다.
  4. AMR이 회전을 마치면 관제탑에 완료를 알리고, 관제탑은 `data = False`를 발행하여 로봇이 반대쪽 5~8번 슬롯에 이어서 적재하도록 해제합니다.
  5. 8번째 상자가 가득 차는 시점에도 동일하게 일시정지(`True`) 신호를 쏴 로봇 팔을 멈추고, AMR이 새 빈 작업대로 교체를 마친 시점에 `False`로 재개시킵니다.

### 📌 3.2 AMR 동시 기동 3대 제한 (뮤텍스 큐) 알고리즘
* **알고리즘 상세**:
  1. 스케줄링 타이머가 Redis ZSET 큐를 주기적으로 확인합니다.
  2. 현재 주행 중인 AMR 태스크 카운터 `active_amr_tasks` 값을 검사합니다.
  3. 카운터 값이 **3 이상**이면, 신규 태스크 배정을 중단하고 큐에 보존한 채 루프를 종료합니다.
  4. 실행 중이던 AMR이 액션 완료(Succeeded/Failed/Canceled)를 보고하면 카운터를 `1` 차감하고, 즉시 다음 대기 큐 명령을 POP하여 다른 AMR에게 하달합니다.

### 📌 3.3 최단 거리 기반 AMR 최적 매핑 알고리즘
* **수식**: Euclidean Distance $d = \sqrt{(x_{start} - x_{amr})^2 + (y_{start} - y_{amr})^2}$
* **알고리즘 구현**: 태스크가 출발할 지점(예: `sg2_in_01_A` = `(7.5, 1.5)`)의 좌표를 DB에서 획득한 후, 현재 `state = 'IDLE'` 이면서 가용한(`available = true`) 모든 AMR의 실시간 Redis 좌표를 조회하여 **거리가 가장 짧은(최소 $d$) AMR**을 찾아 명령을 하달합니다.

In [ ]:
# 최단거리 AMR 매핑 알고리즘을 모방하는 검증 코드를 수행합니다.
import math

# 출발 타겟 위치 (1번 입고 라인 Active)
start_x, start_y = 7.5, 1.5

# Redis에 올라와 있는 AMR들의 가상 상태
amr_states = {
    "AMR_01": {"x": -6.0, "y": -9.0, "state": "IDLE"},
    "AMR_02": {"x": 6.0, "y": 1.5, "state": "IDLE"},
    "AMR_03": {"x": -3.0, "y": 0.0, "state": "BUSY"}, # 바쁨
    "AMR_04": {"x": 7.5, "y": -3.0, "state": "IDLE"},
    "AMR_05": {"x": 0.0, "y": 9.0, "state": "IDLE"}
}

best_amr = None
min_dist = float("inf")

print(f"🎯 작업 발생지 좌표: ({start_x}, {start_y})")
for amr_id, info in amr_states.items():
    if info["state"] == "IDLE":
        dist = math.sqrt((start_x - info["x"])**2 + (start_y - info["y"])**2)
        print(f"   * {amr_id}: 위치 ({info['x']}, {info['y']}) | 거리: {dist:.2f}m")
        if dist < min_dist:
            min_dist = dist
            best_amr = amr_id

print(f"\n🏆 최단 거리 배정 대상 로봇: {best_amr} (거리: {min_dist:.2f}m)")

---

## 💻 4. FastAPI 웹 대시보드 서버 (`dashboard_server.py`)

대시보드는 웹소켓을 통한 실시간 정보 분배 및 DOM 성능 렉 개선을 이룬 중앙 모니터링 허브입니다.

### 📌 4.1 실시간 웹소켓 양방향 통신 시퀀스 흐름

```mermaid
graph TD
    Client[웹 대시보드 브라우저] -->|1. WebSocket 연결 요청 /ws| Dash[FastAPI 백엔드]
    Dash -->|2. Connection 수락 및 리스트 저장| ConnMgr[Connection Manager]
    Timer[1.5초 주기 브로드캐스트 루프] -->|3. PostgreSQL/Redis에서 최신 데이터 조회| Dash
    Dash -->|4. 데이터 JSON 직렬화| ConnMgr
    ConnMgr -->|5. 모든 연결된 클라이언트에 send_text| Client
    Client -->|6. CSS absolute 포지셔닝으로 즉시 렌더링| Client
```

### 📌 4.2 absolute 포지셔닝 기반 경량 렌더링
* **기존 문제**: 2D 캔버스에 수많은 객체를 매 프레임 그리면 브라우저 싱글 스레드 렉이 심해져 화면 제어가 불가능했습니다.
* **해결책**: HTML/CSS 바둑판 배경판 하나만 띄워둔 후, **상대 좌표 변환 공식**을 이용해 31개의 고정 설비와 이동하는 로봇들의 top/left % 좌표만 스타일시트로 동적 조절합니다.
  * **변환 수식**:
    * $Grid_X = 50 + (x \times 5.7)\%$
    * $Grid_Y = 50 - (y \times 5.0)\%$
  * DOM 객체 개수가 크게 줄어 렉 현상이 완전히 사라졌습니다.

In [ ]:
# Redis에 저장된 시스템 플래그 값과 대시보드 락 상태를 검증합니다.
import redis
import os

redis_host = os.environ.get("REDIS_HOST", "localhost")
try:
    r = redis.Redis(host=redis_host, port=6379, decode_responses=True)
    
    # 1. 영업 시작 버튼 해제 조건 점검
    csv_status = r.get("system:csv_loaded") == "true"
    
    # 연결된 AMR 기기 스캔
    amr_keys = r.keys("amr:AMR_*")
    online_amrs = []
    for key in amr_keys:
        state = r.hget(key, "state")
        if state:
            online_amrs.append(key.split(":")[1])
            
    print("🔌 대시보드 영업 기동 조건 자가진단:")
    print(f"   * CSV 파일 업로드 상태: {'초록색 (정상)' if csv_status else '빨간색 (미업로드)'}")
    print(f"   * 실시간 연동 감지된 AMR 대수: {len(online_amrs)}대 ({', '.join(online_amrs) if online_amrs else '없음'})")
    
    is_start_button_unlocked = csv_status and (len(online_amrs) > 0)
    print(f"\n📢 결과: [영업 시작] 버튼 비활성화 해제 가능 여부 -> {'🔓 활성화 (START 가능)' if is_start_button_unlocked else '🔒 비활성화 (락 잠금)'}")
    
except Exception as e:
    print(f"❌ Redis 연결 불가: {e}")